1. Load PyHealth/sample data
2. Build Co-Occurrence Matrix
3. Convert matrix to GloveDataset dataloader that returns i, j, counts, weights
4. Create Keep Model
5. Pass data loader and model to trainer

In [1]:
from collections import defaultdict, Counter
import networkx as nx
import numpy as np
from pyhealth.datasets import OMOPDataset
from pyhealth.models import KeepEmbedding, N2V
from pyhealth.trainer import Trainer
import warnings
import sys
import torch
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cpu


In [2]:
class GloveDataset(Dataset):
    def __init__(self, cooc_matrix, num_words, x_max, alpha):
        super(GloveDataset, self).__init__()
        self.data = []
        for i in range(cooc_matrix.shape[0]):
            for j in range(cooc_matrix.shape[1]):
                if cooc_matrix[i, j] > 0:
                    self.data.append((i, j, cooc_matrix[i, j]))
        self.cooc_matrix = cooc_matrix
        self.num_words = num_words
        self.x_max = x_max
        self.alpha = alpha

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        i, j, count = self.data[idx]
        weight = (count / self.x_max) ** self.alpha if count < self.x_max else 1.0
        # Return dictionary with keys matching KeepEmbedding.forward() parameters
        return {
            "i_indices": torch.tensor(i),
            "j_indices": torch.tensor(j),
            "counts": torch.tensor(count).float(),
            "weights": torch.tensor(weight).float(),
        }
    
def get_code_and_ancestors(graph, code):
    # Start with the code itself
    codes_set = {code}
    
    # Add all ancestors if code exists in graph
    if code in graph:
        ancestors = nx.ancestors(graph, code)
        codes_set.update(ancestors)
    else:
        # Code not in graph - may be rare/invalid, skip silently
        # print(f"Code {code} not found in concept graph (may be rare/invalid)")
        pass
    
    return codes_set

In [3]:
# Load OMOPDataset and extract condition codes for all patients
dataset = OMOPDataset(
    root=r"C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv",
    tables=["condition_occurrence"],
    dataset_name="omop",
    dev=False
)

dataset.stats()

print("Extracting condition codes from all patients...")
        
patient_conditions = defaultdict(list)

# Iterate through all patients
for patient in dataset.iter_patients():
    patient_id = patient.patient_id
    
    # Get all condition events for this patient
    condition_events = patient.get_events(event_type="condition_occurrence")
    
    # Extract condition_concept_id from each event
    for event in condition_events:
        code = event.attr_dict.get("condition_concept_id")
        if code is not None:
            patient_conditions[patient_id].append(code)

print(f"Extracted conditions for {len(patient_conditions)} patients")

No config path provided, using default OMOP config
Initializing omop dataset from C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv (dev mode: False)
No cache_dir provided. Using default cache dir: C:\Users\michi\AppData\Local\pyhealth\pyhealth\Cache\d8eca071-4bf2-5cb6-ba60-95451c941912
Found cached event dataframe: C:\Users\michi\AppData\Local\pyhealth\pyhealth\Cache\d8eca071-4bf2-5cb6-ba60-95451c941912\global_event_df.parquet
Dataset: omop
Dev mode: False
Number of patients: 100
Number of events: 17408
Extracting condition codes from all patients...
Extracted conditions for 100 patients


In [4]:
# Filter out conditions codes with <2 occurrences in patient history
filtered_conditions = {}
        
before_count = sum(len(codes) for codes in patient_conditions.values())
before_unique = len(set(code for codes in patient_conditions.values() for code in codes))

for patient_id, codes in patient_conditions.items():
    # Count occurrences of each code
    code_counts = Counter(codes)
    
    filtered_codes = [code for code, count in code_counts.items() if count >= 2]
    
    if filtered_codes:
        filtered_conditions[patient_id] = filtered_codes

after_count = sum(len(codes) for codes in filtered_conditions.values())
after_unique = len(set(code for codes in filtered_conditions.values() for code in codes))

print(f"Before Filtering:")
print(f"Total codes: {before_count}, Unique: {before_unique}")
print("="*70)
print(f"After Filtering:")
print(f"Total codes: {after_count}, Unique: {after_unique}")

Before Filtering:
Total codes: 16441, Unique: 979
After Filtering:
Total codes: 799, Unique: 280


In [5]:
n2v = N2V()

# Build the concept relationship graph from conditions
print("Building concept relationship graph...")
graph = n2v.create_graph(
    path=r"C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv",
    domain_type=["Condition"]
)

print(f"Graph loaded successfully:")
print(f"  Nodes (unique concepts): {len(graph.nodes())}")
print(f"  Edges (relationships): {len(graph.edges())}")

print("Applying hierarchy roll-up (dense: each code -> itself + all parents)...")

rolled_up_conditions = {}

# Track statistics
total_original_codes = 0
total_rolled_up_codes = 0
patients_processed = 0

for patient_id, codes in filtered_conditions.items():
    expanded_codes = set()
    
    # For each code, add it and all ancestors
    for code in codes:
        code_and_ancestors = get_code_and_ancestors(graph, code)
        expanded_codes.update(code_and_ancestors)
    
    if expanded_codes:
        rolled_up_conditions[patient_id] = expanded_codes
        total_original_codes += len(codes)
        total_rolled_up_codes += len(expanded_codes)
        patients_processed += 1

# Log statistics
print(f"Roll-up complete:")
print(f"  Patients processed: {patients_processed}")
print(f"  Original codes per patient (avg): {total_original_codes / patients_processed:.2f}")
print(f"  Rolled-up codes per patient (avg): {total_rolled_up_codes / patients_processed:.2f}")
print(f"  Expansion factor: {total_rolled_up_codes / total_original_codes:.2f}x")

Building concept relationship graph...
Loading concepts from C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv\2b_concept.csv
Loading concept relationships from C:\Users\michi\Workspace\mimic-iv-demo-data-in-the-omop-common-data-model-0.9\1_omop_data_csv\2b_concept_relationship.csv
Loaded 3885 concepts and 7716 relationships
Filtered to 34 concepts in domains: ['Condition']
Created set of 34 concept IDs
Found 0 relationships between concepts
Graph loaded successfully:
  Nodes (unique concepts): 34
  Edges (relationships): 0
Applying hierarchy roll-up (dense: each code -> itself + all parents)...
Roll-up complete:
  Patients processed: 100
  Original codes per patient (avg): 7.99
  Rolled-up codes per patient (avg): 7.99
  Expansion factor: 1.00x


In [6]:
# Construct co-occurrence matrix for rolled-up conditions
print("Building code index from rolled-up conditions...")
        
# Collect all unique codes after roll-up
unique_codes = set()
for codes in rolled_up_conditions.values():
    unique_codes.update(codes)

# Sort for reproducibility
unique_codes = sorted(unique_codes)

# Create bidirectional mapping
code_to_index = {code: idx for idx, code in enumerate(unique_codes)}
index_to_code = {idx: code for code, idx in code_to_index.items()}


print("Building co-occurrence matrix...")

num_codes = len(code_to_index)

# Initialize matrix
cooc_matrix = np.zeros((num_codes, num_codes), dtype=np.float32)

# Track statistics
total_pairs = 0
patients_processed = 0

for patient_id, codes in rolled_up_conditions.items():
    # Convert codes to indices
    code_indices = [code_to_index[code] for code in codes]
    
    # Create all unique pairs
    for i in range(len(code_indices)):
        for j in range(i + 1, len(code_indices)):
            idx_i = code_indices[i]
            idx_j = code_indices[j]
            
            # Increment both matrix[i,j] and matrix[j,i] for symmetry
            cooc_matrix[idx_i, idx_j] += 1.0
            cooc_matrix[idx_j, idx_i] += 1.0
            
            total_pairs += 1
    
    patients_processed += 1

print(f"Matrix construction complete:")
print(f"  Matrix shape: {cooc_matrix.shape}")
print(f"  Patients processed: {patients_processed}")
print(f"  Total co-occurrence pairs: {total_pairs}")
print(f"  Matrix sparsity: {(cooc_matrix == 0).sum() / cooc_matrix.size * 100:.2f}%")
print(f"  Matrix sum (total co-occurrences): {cooc_matrix.sum():.0f}")
print(f"  Min value: {cooc_matrix.min():.2f}, Max value: {cooc_matrix.max():.2f}")

Building code index from rolled-up conditions...
Building co-occurrence matrix...
Matrix construction complete:
  Matrix shape: (280, 280)
  Patients processed: 100
  Total co-occurrence pairs: 7978
  Matrix sparsity: 84.11%
  Matrix sum (total co-occurrences): 15956
  Min value: 0.00, Max value: 52.00


In [7]:
# Parameters for GloveDataset (standard GloVe hyperparameters)
x_max = 100  # Maximum co-occurrence count before weight saturates
alpha = 0.75  # Weighting factor exponent
batch_size = 128

# Create GloveDataset from co-occurrence matrix
print("Creating GloveDataset from co-occurrence matrix...")
num_words = cooc_matrix.shape[0]
glove_dataset = GloveDataset(cooc_matrix, num_words, x_max, alpha)

print(f"  GloveDataset created:")
print(f"    Num words (diagnosis codes): {num_words}")
print(f"    Dataset size (co-occurrence pairs): {len(glove_dataset)}")

# Create DataLoader
data_loader = DataLoader(glove_dataset, batch_size=batch_size, shuffle=True)
print(f"  DataLoader created: batch_size={batch_size}, num_batches={len(data_loader)}")

# Initialize KeepEmbedding with MATCHING parameters as builder
print("\nInitializing KeepEmbedding with builder parameters...")
keep_model = KeepEmbedding(
    dataset=None,  # GloVe training doesn't require full dataset
    graph = graph,  # Pass the concept relationship graph for Node2Vec
    num_words=num_words,
    embedding_dim=100,
    walk_length=30,
    num_walks=750,
    lambda_reg=1.0e-3,  # Regularization strength (balances GloVe vs graph prior)
    reg_norm=None,  # Use cosine similarity for regularization distance
    log_scale=False,  # No log scaling on regularization
    code_to_index=code_to_index,  # Pass vocabulary mapping to filter embeddings
    device=device
)

print(f"  KeepEmbedding initialized:")
print(f"    Embedding dimension: 100")
print(f"    Regularization lambda: 1.0")
print(f"    Device: {device}")
print(f"\nModel ready for training!")

Creating GloveDataset from co-occurrence matrix...
  GloveDataset created:
    Num words (diagnosis codes): 280
    Dataset size (co-occurrence pairs): 12458
  DataLoader created: batch_size=128, num_batches=98

Initializing KeepEmbedding with builder parameters...
Initializing Node2Vec with embedding_dim=100...
Graph created with 34 nodes and 0 edges
Initializing Node2Vec with embedding_dim=100 walk_length=30, num_walks=750


Computing transition probabilities:   0%|          | 0/34 [00:00<?, ?it/s]

Creating embedding vectors for 34 concepts...
Embedding matrix shape: (34, 100)
Created embedding matrix with shape: (34, 100)
Filtering embeddings from 34 concepts to 280 vocabulary items...
  280/280 codes not found in graph, using mean embedding as fallback
Filtered embedding matrix shape: (280, 100)
Initialized KEEP Embedding with 280 tokens
Embedding dimension: 100
Regularization lambda: 0.001
Regularization norm: None
Log scaling: False
  KeepEmbedding initialized:
    Embedding dimension: 100
    Regularization lambda: 1.0
    Device: cpu

Model ready for training!


In [8]:
print("Training KeepEmbedding model with GloVe objective + graph regularization...")
print(f"  Total batches: {len(data_loader)}")
print(f"  Training for 50 epochs\n")

trainer = Trainer(model=keep_model)

# Train with GloVe objective + Node2Vec graph regularization
trainer.train(
    train_dataloader=data_loader,
    val_dataloader=None,  # No validation split for embedding training
    epochs=50,
    optimizer_class=torch.optim.Adam,
    optimizer_params={"lr": 0.01},  # Learning rate via optimizer_params
    monitor="loss",  # Monitor training loss
    patience=10,  # Early stopping patience
)

print("\nTraining complete!")
print(f"Learned embeddings shape: {keep_model.embeddings_v.weight.shape}")
print(f"Ready for downstream tasks!")

Training KeepEmbedding model with GloVe objective + graph regularization...
  Total batches: 98
  Training for 50 epochs

KeepEmbedding(
  (embeddings_v): Embedding(280, 100)
  (embeddings_u): Embedding(280, 100)
  (biases_v): Embedding(280, 1)
  (biases_u): Embedding(280, 1)
)
Metrics: None
Device: cpu

Training:
Batch size: 128
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.01}
Weight decay: 0.0
Max grad norm: None
Val dataloader: None
Monitor: loss
Monitor criterion: max
Epochs: 50
Patience: 10



Epoch 0 / 50:   0%|          | 0/98 [00:00<?, ?it/s]

--- Train epoch-0, step-98 ---
loss: 0.7966



Epoch 1 / 50:   0%|          | 0/98 [00:00<?, ?it/s]

--- Train epoch-1, step-196 ---
loss: 0.5038



...

--- Train epoch-47, step-4704 ---
loss: 0.1948



Epoch 48 / 50:   0%|          | 0/98 [00:00<?, ?it/s]

--- Train epoch-48, step-4802 ---
loss: 0.2131



Epoch 49 / 50:   0%|          | 0/98 [00:00<?, ?it/s]

--- Train epoch-49, step-4900 ---
loss: 0.2149

Training complete!
Learned embeddings shape: torch.Size([280, 100])
Ready for downstream tasks!
